# Gridalytics - Data Exploration

Explore the demand, weather, and holiday data in the Gridalytics database.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from datetime import date

from src.data.db.session import get_session
from src.data.loaders import load_demand, load_weather, get_data_summary

pd.set_option('display.max_columns', 20)
%matplotlib inline

## 1. Data Summary

In [ ]:
with get_session() as session:
    summary = get_data_summary(session)

for k, v in summary.items():
    print(f'{k}: {v}')

## 2. Demand Data

In [ ]:
with get_session() as session:
    hourly = load_demand(session, 'hourly', date(2024, 1, 1), date(2026, 3, 28))

print(f'Shape: {hourly.shape}')
print(f'Date range: {hourly.index.min()} to {hourly.index.max()}')
hourly['delhi_mw'].describe()

In [ ]:
# Demand over time
fig = px.line(hourly.resample('D').mean(), y='delhi_mw', title='Daily Average Demand (MW)')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

In [ ]:
# Daily demand pattern by hour
hourly['hour'] = hourly.index.hour
hourly_pattern = hourly.groupby('hour')['delhi_mw'].agg(['mean', 'std'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=hourly_pattern.index, y=hourly_pattern['mean'], name='Mean', line=dict(color='#3b82f6')))
fig.add_trace(go.Scatter(x=hourly_pattern.index, y=hourly_pattern['mean'] + hourly_pattern['std'], name='+1 std', line=dict(dash='dash', color='gray')))
fig.add_trace(go.Scatter(x=hourly_pattern.index, y=hourly_pattern['mean'] - hourly_pattern['std'], name='-1 std', line=dict(dash='dash', color='gray')))
fig.update_layout(title='Hourly Demand Pattern', xaxis_title='Hour of Day', yaxis_title='Demand (MW)', template='plotly_dark')
fig.show()

## 3. Weather Data

In [ ]:
with get_session() as session:
    weather = load_weather(session, date(2024, 1, 1), date(2026, 3, 28))

print(f'Shape: {weather.shape}')
weather.describe().round(1)

In [ ]:
# Temperature vs Demand scatter
merged = hourly[['delhi_mw']].join(weather[['temperature_2m']], how='inner')
fig = px.scatter(merged.sample(5000), x='temperature_2m', y='delhi_mw', 
                 opacity=0.3, title='Temperature vs Demand',
                 labels={'temperature_2m': 'Temperature (C)', 'delhi_mw': 'Demand (MW)'})
fig.update_layout(template='plotly_dark')
fig.show()

## 4. Seasonal Patterns

In [ ]:
daily = hourly.resample('D').mean()
daily['month'] = daily.index.month
daily['season'] = pd.cut(daily['month'], bins=[0,2,4,6,9,10,12], 
                        labels=['Winter','Spring','Summer','Monsoon','Autumn','Winter2'])

fig = px.box(daily, x='season', y='delhi_mw', title='Demand by Season',
             labels={'delhi_mw': 'Daily Avg Demand (MW)'})
fig.update_layout(template='plotly_dark')
fig.show()